In [ ]:
import os
import pickle 
from tqdm.auto import tqdm
import pandas as pd
import torch

In [ ]:
from langchain.chains import GraphCypherQAChain
from langchain.graphs import Neo4jGraph
from langchain_core.language_models import BaseLanguageModel 
from langchain_neo4j.chains.graph_qa.cypher_utils import CypherQueryCorrector
import ast
import json

def evaluate_model_qa_chain(model_name, model: BaseLanguageModel, dataset):
    '''
    Evaluate the selected model on the given dataset, using GraphCypherQAChain
    '''
    torch.cuda.empty_cache()

    # Load precomputed schema from file
    with open('../precomputed/KG_schemas/kg_structured_schema.json', 'r') as f:
        all_schemas = json.load(f)
    schema = all_schemas.get("neo4jcopy") # get schema from precomputed

    if not schema:
        raise ValueError("Schema for 'neo4jcopy' not found in precomputed/KG_schemas_kg_schema.json")

    if dataset == 'sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx', sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'

    output_dir = f"Results/{subfolder}/{model_name}/QAChain"
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}.xlsx")

    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:
        df_results = df.copy()
        df_results['generated_cypher'] = None
        df_results['answer'] = None

    print(f'Starting evaluation for {model_name} on {dataset}!!!!!!!!!!!')

    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i, question)

            # Load DB credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')

            # for QAChain!!!!!
            graph = Neo4jGraph(
                    url=database_url,
                    username=database_user,
                    password=database_password,
            )

            corrector = CypherQueryCorrector(schema)

            chain = GraphCypherQAChain.from_llm(
                    llm=model,
                    graph=graph,
                    cypher_corrector=corrector,
                    verbose=False,
                    return_intermediate_steps=True,
                    allow_dangerous_requests=True
                )

            result = None
            try:
                result = chain.invoke({"query": question})
                df_results.at[i, 'output'] = str(result.get('result', ''))
                df_results.at[i, 'answer'] = result.get('result', '')
                df_results.at[i, 'path'] = str(result.get('intermediate_steps', []))
            except Exception as e: # if qachain doesn't return any results (mostly syntax errors)
                print(f"Error in row {i}: {e}")
                try:
                    print("\n-- Attempted result before error --")
                    print(result)
                    print("-- End of attempted result --\n")
                except:
                    pass
            finally:
                generated_query = None # keep generated query from whatever possible
                if result:
                    generated_query = result.get('cypher_query', None)

                    if not generated_query: # if cypher_query failed, get it from intermediate steps
                        intermediate_steps = result.get('intermediate_steps', [])
                        for step in intermediate_steps:
                            if isinstance(step, dict) and 'query' in step: # if generated steps exist
                                generated_query = step['query']
                                break
                df_results.at[i, 'generated_cypher'] = generated_query if generated_query is not None else 'None!!' # None!! if nothing was generated
        else:
            print(f"Skipping row {i}, already computed.")

    return df_results

In [3]:
from Configuration.models import models #import models to evaluate
model_name = 'gpt_4o'
df_results = evaluate_model_qa_chain(model_name, models[model_name](), 'sustain')

Starting evaluation for gpt_4o on sustain!!!!!!!!!!!


  0%|          | 0/52 [00:00<?, ?it/s]

Skipping row 0, already computed.
Skipping row 1, already computed.
Skipping row 2, already computed.
Skipping row 3, already computed.
Skipping row 4, already computed.
Skipping row 5, already computed.
Skipping row 6, already computed.
Skipping row 7, already computed.
Skipping row 8, already computed.
Skipping row 9, already computed.
Skipping row 10, already computed.
Skipping row 11, already computed.
Skipping row 12, already computed.
Skipping row 13, already computed.
Skipping row 14, already computed.
Skipping row 15, already computed.
Skipping row 16, already computed.
Skipping row 17, already computed.
Skipping row 18, already computed.
19 Which area names have EUcode as property? If yes, return Property exists. Otherwise Property does not exist


/tmp/ipykernel_497819/4270053171.py:78: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


Error in row 19: {code: Neo.ClientError.Statement.SyntaxError} {message: The property existence syntax `... exists(variable.property)` is no longer supported. Please use `variable.property IS NOT NULL` instead. (line 3, column 18 (offset: 39))
"RETURN CASE WHEN exists(a.EUcode) THEN 'Property exists' ELSE 'Property does not exist' END AS result"
                  ^}

-- Attempted result before error --
None
-- End of attempted result --

Skipping row 20, already computed.
Skipping row 21, already computed.
Skipping row 22, already computed.
Skipping row 23, already computed.
Skipping row 24, already computed.
Skipping row 25, already computed.
Skipping row 26, already computed.
Skipping row 27, already computed.
Skipping row 28, already computed.
Skipping row 29, already computed.
Skipping row 30, already computed.
Skipping row 31, already computed.
Skipping row 32, already computed.
Skipping row 33, already computed.
Skipping row 34, already computed.
Skipping row 35, already computed

In [4]:
df_results

,Unnamed: 0,Level,Sublevel,Level of Difficulty,Topic,Question,"Cypher query ( Single Match, integrated properties)","Cypher query 2 (Single Match, Where)","Cypher query 3 (Multiple Match, integrated properties)","Cypher query 4 (Multiple Match,Where)",Alternative query,Subschema,generated_cypher,answer,output,path
0,0,Level A: Explict Facts,Sublevel 1:Nodes,Easy,SDG,What is the title of the goal with code 2?,MATCH (n:Goal{code : '2' })RETURN n.title,MATCH (n:Goal) WHERE n.code = '2' RETURN n.title,NaN,NaN,NaN,"Goal {title: STRING, description: STRING, code...","MATCH (g:Goal {code: ""2""}) RETURN g.title","The title of the goal with code 2 is ""End hung...","The title of the goal with code 2 is ""End hung...","[{'query': 'cypher\nMATCH (g:Goal {code: ""2""})..."
1,1,Level A: Explict Facts,Sublevel 1:Nodes,Easy,GeoArea,Which is the EUcode for the geoarea with name ...,MATCH (g:GeoArea{name:'Greece'}) RETURN g.EUcode,MATCH (g:GeoArea) WHERE g.name='Greece' RETURN...,NaN,NaN,NaN,"GeoArea {name: STRING, M49code: STRING, ISOalp...","MATCH (g:GeoArea {name: ""Greece""}) RETURN g.EU...",The EUcode for the geoarea with the name Greec...,The EUcode for the geoarea with the name Greec...,"[{'query': 'cypher\nMATCH (g:GeoArea {name: ""G..."
2,2,Level A: Explict Facts,Sublevel 1:Nodes,Easy,GeoArea,Which is the name of the area that has the EUc...,MATCH (a:Area{EUcode: 'FR'}) RETURN a.name,MATCH (a:Area) WHERE a.EUcode= 'FR' RETURN a....,NaN,NaN,NaN,"Area {name: STRING, M49code: STRING, ISOalpha2...",MATCH (a:Area {EUcode: 'FR'}) RETURN a.name,The name of the area that has the EUcode FR is...,The name of the area that has the EUcode FR is...,"[{'query': ""cypher\nMATCH (a:Area {EUcode: 'FR..."
3,3,Level A: Explict Facts,Sublevel 1:Nodes,Easy,CS,Which is the title of case study with code cs_1?,MATCH (c:CaseStudy{code:'cs_1'}) RETURN c.title,MATCH (c:CaseStudy) WHERE c.code = 'cs_1' RETU...,NaN,NaN,NaN,"CaseStudy {title: STRING, description: STRING,...","MATCH (cs:CaseStudy {code: ""cs_1""}) RETURN cs....",The title of the case study with code cs_1 is ...,The title of the case study with code cs_1 is ...,[{'query': 'cypher\nMATCH (cs:CaseStudy {code:...
4,4,Level A: Explict Facts,Sublevel 1:Nodes,Easy,Hazard,Which are the hazards names with Environmental...,MATCH (h:Hazard{type:'Environmental'})\nRETURN...,MATCH (h:Hazard) WHERE h.type = 'Environmental...,NaN,NaN,NaN,"Hazard {name: STRING, cluster: STRING, type: S...","MATCH (h:Hazard {type: ""Environmental""}) RETUR...",Air Pollution (Outdoor - Result Of Product Com...,Air Pollution (Outdoor - Result Of Product Com...,"[{'query': 'cypher\nMATCH (h:Hazard {type: ""En..."
5,5,Level B: Implicit Facts,Sublevel 1:One-hop,Easy,"Target, SDG",What are the target codes that goal with code ...,MATCH (n:Goal{code: '8' })-[:HAS_TARGET]->(t:T...,MATCH (n:Goal)-[:HAS_TARGET]->(t:Target)\nWHER...,NaN,NaN,NaN,"Goal {title: STRING, description: STRING, code...",MATCH (g:Goal {code: '8'})-[:HAS_TARGET]->(t:T...,The target codes that goal with code 8 has are...,The target codes that goal with code 8 has are...,"[{'query': ""cypher\nMATCH (g:Goal {code: '8'})..."
6,6,Level B: Implicit Facts,Sublevel 1:One-hop,Easy,"Indicator, Series",Which are the series' code and description tha...,MATCH (i:Indicator{code:'vulnerability'})-[:HA...,MATCH (i:Indicator)-[:HAS_SERIES]->(s:Series)\...,NaN,NaN,NaN,"Indicator {description: STRING, code: STRING}-...",MATCH (i:Indicator {code: 'vulnerability'})-[:...,The series' code and description for the indic...,The series' code and description for the indic...,"[{'query': ""cypher\nMATCH (i:Indicator {code: ..."
7,7,Level B: Implicit Facts,Sublevel 1:One-hop,Easy,"Indicator, PolicyTarget",Which indicator codes have EU policy targets ?,MATCH (i:Indicator)-[:HAS_POLICY_TARGET]->(:EU...,NaN,NaN,NaN,NaN,"Indicator {description: STRING, code: STRING}-...",MATCH (i:Indicator)-[:HAS_POLICY_TARGET]->(:EU...,The indicator codes that have EU policy target...,The indicator codes that have E